### Make sure to run this notebook on a decent computer, with ollama installed

In [ ]:
import pandas as pd

from slrrag.input import FIRST_ROUND_QUESTIONS
from slrrag.settings import PATH_CSV_02_FILTERED_TOPIC, PATH_CSV_03_AGENTS_ANWSERS
from slrrag.toolbox.toolbox_notebooks import plot_histogram

%load_ext autoreload
%autoreload 2
from slrrag.ollama_analysis import analyze_papers_with_list_agents, DEFAULT_PROMPT

ollama_models = [
    "granite4.1:30b",
    "ministral-3:14b",
    "llama3.1:8b",
]

In [3]:
paper_hard = [
    "a-collision-risk-based-method-for-safety-distance-recommendation-for-offshore-platforms-and-navigation-routes-considering-ships-out-of-control",
    "a-deep-learning-method-for-ship-detection-and-traffic-monitoring-in-an-offshore-wind-farm-area",
    "a-dynamic-risk-assessment-method-to-address-safety-of-navigation-concerns-around-offshore-renewable-energy-installations",
    "a-fuzzy-madm-based-approach-for-site-selection-of-offshore-wind-farm-in-busy-waterways-in-china",
    "a-hierarchical-path-planning-framework-for-large-scale-sparse-environments-layered-grid-refinement-and-bidirectional-shortcuts-with-application-to-offshore-wind-farm-inspection",
    "a-marine-gis-based-alert-system-to-prevent-vessels-collision-with-offshore-platforms",
    "a-multi-criteria-analysis-framework-for-conflict-resolution-in-the-case-of-offshore-wind-farm-sitting-a-study-of-england-and-the-netherlands-offshore-space",
    "a-new-framework-for-assessment-of-offshore-wind-farm-location",
    "a-novel-fuzzy-bayesian-network-based-madm-model-for-offshore-wind-turbine-selection-in-busy-waterways-an-application-to-a-case-in-china",
    "a-novel-fuzzy-multi-factor-navigational-risk-assessment-method-for-ship-route-optimization-in-costal-offshore-wind-farm-waters",
    "a-novel-method-for-navigational-risk-assessment-in-wind-farm-waters-based-on-the-fuzzy-inference-system",
    "a-path-planning-approach-based-on-multi-direction-a-algorithm-for-ships-navigating-within-wind-farm-waters",
    "a-review-on-the-impact-of-offshore-wind-farm-development-on-maritime-traffic-safety-and-the-improvement-of-traffic-management-and-regulatory-systems",
    "a-study-on-the-design-of-coastal-fairway-width-based-on-a-risk-assessment-model-in-korean-waterways",
    "a-study-on-the-factors-affecting-optimal-site-of-offshore-wind-farm-from-the-perspective-of-maritime-traffic-using-spatial-analysis",
    "a-study-on-traffic-safety-assessments-for-fishing-vessels-near-the-southwest-sea-offshore-wind-farm",
    "a-system-theory-and-complex-network-fused-approach-to-analyze-vesselwind-turbine-allisions-in-offshore-wind-farm-waters",
    "ais-based-characterization-of-navigation-conflicts-along-the-us-atlantic-coast-prior-to-development-of-wind-energy",
    "analysis-of-navigational-risk-indicators-as-a-function-of-the-ship's-domain-width-for-the-selected-offshore-wind-farm-in-the-baltic-sea",
    "analysis-of-the-impact-of-wind-farms-on-the-detection-performance-of-marine-radar-a-case-study-of-the-taizhou-no1-offshore-wind-farm",
    "anti-collision-warning-of-offshore-wind-farm-based-on-kalman-filter",
    "applying-the-fuzzy-distinction-method-to-analyze-the-precautionary-area-of-the-wind-farm-along-the-western-coast-of-taiwan",
    "assessing-the-impacts-to-vessel-traffic-from-offshore-wind-farms-in-the-thames-estuary",
    "assessing-the-validity-of-navigation-risk-assessments-a-study-of-offshore-wind-farms-in-the-uk",
    "assessment-of-the-influence-of-offshore-wind-farms-on-ship-traffic-flow-based-on-ais-data",
    "changes-in-maritime-traffic-patterns-according-to-installation-of-floating-lidar-using-spatial-analysis",
    "chapitre-7-faire-de-l'affaire-des-simulateurs-pour-les-installations-d'énergie-renouvelable-extracôtières-évaluation-des-risques-liés-à-la-navigation",
    "collision-risk-modelling-of-supply-vessels-and-offshore-platforms-under-uncertainty",
    "comparison-of-fuzzy-bayesian-network-methods-for-maritime-applications",
    "comprehensive-assessment-model-for-navigation-safety-risks-in-area-of-offshore-wind-farm",
    "dijkstra-and-a-algorithms-for-algorithmic-optimization-of-maritime-routes-and-logistics-of-offshore-wind-farms",
    "enhancing-vessel-operational-safety-and-security-through-cross-layer-data-fusion-a-reliabilityoriented-maritime-surveillance-framework",
    "evaluation-of-ship-importance-in-offshore-wind-farm-area-based-on-fusion-gravity-model-in-complex-network",
    "extracting-the-maritime-traffic-route-in-korea-based-on-probabilistic-approach-using-automatic-identification-system-big-data",
    "fuzzy-based-risk-assessment-and-collision-avoidance-strategy-for-unmanned-surface-vessels-near-offshore-wind-farm",
    "identification-of-navigational-risks-associated-with-wind-farms",
    "improving-the-coexistence-of-offshore-wind-farms-and-shipping-an-international-comparison-of-navigational-risk-assessment-processes",
    "incorporation-of-fault-tree-analysis-and-evidential-reasoning-in-navigational-risk-assessment-of-offshore-wind-farms",
    "intelligent-anomaly-detection-framework-for-ship-navigation-using-multi-source-heterogeneous-data",
    "intelligent-maritime-vessel-detection-and-classification-toward-offshore-wind-energy-zone-surveillance-using-yolov11-and-explainable-hybrid-machine-learning-for-autonomous-maritime-monitoring",
    "intelligent-ship-anchorage-management-a-novel-obs-mlp-fusion-framework-with-multi-modal-feature-decoupling-for-multi-vessel-collaborative-optimization",
    "investigation-on-improving-strategies-for-navigation-safety-in-the-offshore-wind-farm-in-taiwan-strait",
    "likelihood-of-a-marine-vessel-accident-from-wind-energy-development-in-the-atlantic",
    "matrix-a-multidimensional-maritime-traffic-complexity-evaluation-framework",
    "method-for-prediction-of-ship-traffic-behaviour-and-encounter-frequency",
    "multi-criteria-evaluation-of-wave-energy-projects-on-the-south-east-australian-coast",
    "multi-objective-path-planning-for-offshore-wind-farm-vessels-based-on-a-nsga-iii",
    "navigation-safety-evaluation-of-offshore-wind-farm-waterway-based-on-variation-coefficient-cloud-matter-element-model",
    "navigational-safety-hazards-posed-by-offshore-wind-farms-a-comprehensive-literature-review-and-bibliometric-analysis",
    "offshore-wind-farm-siting-method-based-on-multi-source-data-fusion",
    "quantifying-navigational-safety-around-planned-human-modified-environment-with-a-multi-agent-simulator-for-multiple-perspectives",
    "realising-advanced-risk-assessment-of-vessel-traffic-flows-near-offshore-wind-farms",
    "research-on-operation-safety-of-offshore-wind-farms",
    "research-on-the-impact-of-deep-sea-offshore-wind-farms-on-maritime-safety",
    "review-of-probabilistic-risk-assessment-models-for-ship-collisions-with-structures",
    "risk-assessment-model-for-defining-minimum-distance-of-shipping-lanes-from-offshore-wind-farm-location",
    "safe-navigation-distance-between-marine-routes-and-aquaculture-farms-in-south-korea-using-gaussian-mixture-model",
    "safety-assessment-of-offshore-wind-farms-and-ship-navigation-based-on-collision-risk-model",
    "smart-sea-ship-collision-avoidance-of-stationary-structures-through-integrated-machine-learning-radar-image-detection-and-high-fidelity-maneuvering-models",
    "study-on-vessel-traffic-risk-assessment-according-to-waterway-patterns-in-a-southwest-offshore-wind-farm",
    "the-fuzzy-analysis-of-western-windfarm-transportation-safety",
    "the-impact-of-offshore-wind-farm-construction-on-maritime-traffic-complexity-an-empirical-analysis-of-the-yangtze-river-estuary",
    "the-methodology-for-assessing-the-impact-of-offshore-wind-farms-on-navigation-based-on-the-automatic-identification-system-historical-data",
    "the-probability-of-ship-collision-during-the-fully-submerged-towing-process-of-floating-offshore-wind-turbines",
    "using-deep-reinforcement-learning-for-autonomous-vessel-path-planning-in-offshore-wind-farms",
    "vessel-intrusion-interception-utilising-unmanned-surface-vehicles-for-offshore-wind-farm-asset-protection",
    "vessel-traffic-behaviour-near-newly-established-offshore-wind-farms-based-on-ais-data-analysis",
    "vulnerability-and-risk-assessment-of-the-coexistence-of-oil-and-fishing-industries-in-the-southern-gulf-of-mexico",
]

In [10]:
df_03_agents_analysis = pd.read_csv(PATH_CSV_02_FILTERED_TOPIC)

#df_03_agents_analysis = df_03_agents_analysis[df_03_agents_analysis["title_slug"].isin(paper_hard)]
df_03_agents_analysis

,title_slug,is_chosen_first_nra,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,...,keywords_openalex,id_dimensions,topic_dimensions,keywords_dimensions,in_scopus,in_openalex,in_dimensions,is_found,language_scopus,topic_scopus
0,2-year-review-of-project-to-repurpose-steel-tu...,False,2-Year Review of Project To Repurpose Steel Tu...,10.2118/0726-0016-jpt,article,2026,2026-07-01,"_ This article, written by JPT Technology Edit...",NaN,NaN,...,Nuclear decommissioning|Artificial reef|Safety...,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN,False,True,True,True,NaN,NaN
1,2nd-congress-on-motion-sickness-august-12th-15...,False,"2nd Congress on Motion Sickness, August 12th- ...",10.4081/ejtm.2025.14424,article,2025,2025-10-06,"Relevance of motion sickness Motion sickness, ...",NaN,NaN,...,Motion sickness|Motion (physics)|Virtual reali...,NaN,NaN,NaN,False,True,False,True,NaN,NaN
2,3-dof-cooperative-control-of-an-offshore-platf...,False,3-DOF cooperative control of an offshore platf...,10.1016/j.oceaneng.2025.124110,article,2026,2026-03-01,This paper presents a novel Multi-cable Driven...,2-s2.0-105029681916,Adaptive sliding mode control | Attitude contr...,...,Payload (computing)|Control theory (sociology)...,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN,True,True,True,True,eng,Environmental Engineering|Ocean Engineering
3,4d-geomechanical-analysis-to-support-field-str...,False,4D Geomechanical Analysis to Support Field Str...,10.4043/32778-MS,conference-paper,2023,2023-01-01,Fault reactivation due to reservoir depletion ...,2-s2.0-85175582345,NaN,...,Overburden|Geology|Fault (geology)|Submarine p...,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN,True,True,True,True,eng,Energy Engineering and Power Technology|Mechan...
4,978-981-97-7255-1pdf,False,978-981-97-7255-1.pdf,NaN,article,2024,2024-10-21,NaN,NaN,NaN,...,Standardization|Field (mathematics)|Nondestruc...,NaN,NaN,NaN,False,True,False,True,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
721,“dont-blast”-blast-in-place-bip-operations-of-...,False,“Don’t Blast”: blast-in-place (BiP) operations...,10.1007/s00204-020-02743-0,article,2020,2020-06-01,The seas worldwide are threatened by a “new” s...,2-s2.0-85083827767,Biomonitoring | Blast-in-place operations | Bl...,...,NaN,https://app.dimensions.ai/details/publication/...,B12 Engineering,Animals; Biological Monitoring; Consumer Produ...,True,False,True,True,eng,"Toxicology|Health, Toxicology and Mutagenesis"
722,交通流解析に基づく洋上発電設備への衝突危険性の検討,False,交通流解析に基づく洋上発電設備への衝突危険性の検討,10.9749/jin.136.80,article,2017,2017-07-01,A variety of offshore power generation install...,NaN,NaN,...,NaN,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN,False,False,True,True,NaN,NaN
723,넙치-양식장-위해요소중점관리기준haccp-적용모델-개발,False,넙치 양식장 위해요소중점관리기준(HACCP) 적용모델 개발,NaN,article,2013,2013-10-01,We attempted to apply the HACCP system adopted...,NaN,NaN,...,Critical control point|Hazard analysis and cri...,NaN,NaN,NaN,False,True,False,True,NaN,NaN
724,서남해-해상풍력발전단지-내-항로형태에-따른-선박통항-위험도-평가에-관한-연구,False,서남해 해상풍력발전단지 내 항로형태에 따른 선박통항 위험도 평가에 관한 연구,NaN,article,2019,2019-10-31,Domestic southwest offshore wind farms have pr...,NaN,NaN,...,Submarine pipeline|Traffic volume|Offshore win...,NaN,NaN,NaN,False,True,False,True,NaN,NaN


In [11]:
prompt ="""
You are an expert in nautical safety, especially about assessing risk assessment of collision between vessels and offshore platforms. You are conducting a literature review. 
Your task is to evaluate studies based on specific crtiteria.

Inclusion Criteria:
- Goals : Papers with at least one of these five goals ; 1) Quantitative risk assessment of collision and allision between vessels and offshore infrastructures. 2) Safe distance between offshore infrastructures and maritime traffic. 3) Decision making of location of infrastructures or maritime routes, taking account of maritime traffic safety. 4) Operational risk assessment of collision in real-time. 5) Impact of offshore infrastructure on maritime traffic.
- Environment: The context must be related to an offshore infrastructure, like oil platform or wind farms, or other similar infrastructures.

Exclusion Criteria:
- The paper does respond to all the inclusion criteria
- The paper is not in english
- The paper analyzes the consequence of the collision on the infrastructures/ships, but does not study its likelihood.
- The paper talks about other risks than collisions between vessels and offshore infrastructures.

Answer with a JSON object containing:
- 'answer': 'True' or 'False'
- 'explanation': a short explanation (1-2 sentences)

Title: {title}
Abstract: {abstract}
"""

print(
    f"Analizing with models {ollama_models} the following prompt \n------\n{prompt}\n------\n"
)
df_resuls_ollama_agent = analyze_papers_with_list_agents(
    ollama_models, df_03_agents_analysis, prompt
)


Analizing with models ['granite4.1:30b', 'ministral-3:14b', 'llama3.1:8b'] the following prompt 
------

You are an expert in nautical safety, especially about assessing risk assessment of collision between vessels and offshore platforms. You are conducting a literature review. 
Your task is to evaluate studies based on specific crtiteria.

Inclusion Criteria:
- Goals : Papers with at least one of these five goals ; 1) Quantitative risk assessment of collision and allision between vessels and offshore infrastructures. 2) Safe distance between offshore infrastructures and maritime traffic. 3) Decision making of location of infrastructures or maritime routes, taking account of maritime traffic safety. 4) Operational risk assessment of collision in real-time. 5) Impact of offshore infrastructure on maritime traffic.
- Environment: The context must be related to an offshore infrastructure, like oil platform or wind farms, or other similar infrastructures.

Exclusion Criteria:
- The paper d

  0%|          | 0/726 [00:00<?, ?it/s]

100%|██████████| 726/726 [25:01<00:00,  2.07s/it]


----- ministral-3:14b -------


100%|██████████| 726/726 [21:02<00:00,  1.74s/it]


----- llama3.1:8b -------


100%|██████████| 726/726 [11:07<00:00,  1.09it/s]


In [12]:
df_resuls_ollama_agent

,title_slug,is_chosen_first_nra,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,...,in_dimensions,is_found,language_scopus,topic_scopus,answer_granite4.1:30b,explanation_granite4.1:30b,answer_ministral-3:14b,explanation_ministral-3:14b,answer_llama3.1:8b,explanation_llama3.1:8b
0,2-year-review-of-project-to-repurpose-steel-tu...,False,2-Year Review of Project To Repurpose Steel Tu...,10.2118/0726-0016-jpt,article,2026,2026-07-01,"_ This article, written by JPT Technology Edit...",NaN,NaN,...,True,True,NaN,NaN,False,The paper focuses on repurposing decommissione...,False,This paper focuses on **repurposing decommissi...,False,The paper does not meet the inclusion criteria...
1,2nd-congress-on-motion-sickness-august-12th-15...,False,"2nd Congress on Motion Sickness, August 12th- ...",10.4081/ejtm.2025.14424,article,2025,2025-10-06,"Relevance of motion sickness Motion sickness, ...",NaN,NaN,...,False,True,NaN,NaN,False,The paper does not address any aspects of coll...,False,This paper does not meet the inclusion criteri...,False,This paper does not address the goals of risk ...
2,3-dof-cooperative-control-of-an-offshore-platf...,False,3-DOF cooperative control of an offshore platf...,10.1016/j.oceaneng.2025.124110,article,2026,2026-03-01,This paper presents a novel Multi-cable Driven...,2-s2.0-105029681916,Adaptive sliding mode control | Attitude contr...,...,True,True,eng,Environmental Engineering|Ocean Engineering,False,The paper focuses on developing a control syst...,False,This paper focuses on **control systems for pa...,False,The paper does not relate to the assessment of...
3,4d-geomechanical-analysis-to-support-field-str...,False,4D Geomechanical Analysis to Support Field Str...,10.4043/32778-MS,conference-paper,2023,2023-01-01,Fault reactivation due to reservoir depletion ...,2-s2.0-85175582345,NaN,...,True,True,eng,Energy Engineering and Power Technology|Mechan...,False,The paper focuses on geomechanical analysis an...,False,This paper focuses on **geomechanical risk ass...,False,The paper does not meet the inclusion criteria...
4,978-981-97-7255-1pdf,False,978-981-97-7255-1.pdf,NaN,article,2024,2024-10-21,NaN,NaN,NaN,...,False,True,NaN,NaN,False,The provided information (title and abstract) ...,False,The provided title does not include a clear ab...,False,The title does not provide enough information ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
721,“dont-blast”-blast-in-place-bip-operations-of-...,False,“Don’t Blast”: blast-in-place (BiP) operations...,10.1007/s00204-020-02743-0,article,2020,2020-06-01,The seas worldwide are threatened by a “new” s...,2-s2.0-85083827767,Biomonitoring | Blast-in-place operations | Bl...,...,True,True,eng,"Toxicology|Health, Toxicology and Mutagenesis",False,The paper focuses on environmental hazards fro...,False,The paper focuses on the **environmental and c...,False,The paper does not meet the inclusion criteria...
722,交通流解析に基づく洋上発電設備への衝突危険性の検討,False,交通流解析に基づく洋上発電設備への衝突危険性の検討,10.9749/jin.136.80,article,2017,2017-07-01,A variety of offshore power generation install...,NaN,NaN,...,True,True,NaN,NaN,False,The paper does not meet the inclusion criteria...,False,The paper is not in English (title and abstrac...,False,This paper does not meet the inclusion criteri...
723,넙치-양식장-위해요소중점관리기준haccp-적용모델-개발,False,넙치 양식장 위해요소중점관리기준(HACCP) 적용모델 개발,NaN,article,2013,2013-10-01,We attempted to apply the HACCP system adopted...,NaN,NaN,...,False,True,NaN,NaN,False,The paper focuses on developing an HACCP model...,False,The paper focuses on **HACCP (Hazard Analysis ...,False,This paper does not relate to the assessment o...
724,서남해-해상풍력발전단지-내-항로형태에-따른-선박통항-위험도-평가에-관한-연구,False,서남해 해상풍력발전단지 내 항로형태에 따른 선박통항 위험도 평가에 관한 연구,NaN,article,2019,2019-10-31,Domestic southwest offshore wind farms have pr...,NaN,NaN,...,False,True,NaN,NaN,False,The paper does not meet the inclusion criteria...,False,The paper is **not in English** (title and abs...,False,The p

In [13]:
df_resuls_ollama_agent.to_csv(PATH_CSV_03_AGENTS_ANWSERS, index=False)

### Prelimnary results with agents

In [4]:
df_03_agents_analysis = pd.read_csv(PATH_CSV_03_AGENTS_ANWSERS)
len(df_03_agents_analysis), df_03_agents_analysis.dtypes

(726,
 title_slug                         str
 is_chosen_first_nra               bool
 title                              str
 doi                                str
 type                               str
 year                             int64
 date                               str
 abstract                           str
 id_scopus                          str
 keywords_scopus                    str
 relevance_order_scopus         float64
 is_reference                      bool
 id_openalex                        str
 relevance_openalex             float64
 language_openalex                  str
 pdf                                str
 topic_openalex                     str
 fields_openalex                    str
 subfields_openalex                 str
 domain_openalex                    str
 keywords_openalex                  str
 id_dimensions                      str
 topic_dimensions                   str
 keywords_dimensions                str
 in_scopus                        

In [5]:
type_list_col_answer = [
    "answer_" + x for x in ollama_models for q in FIRST_ROUND_QUESTIONS
]
type_list_col_explanation = [
    "explanation_" + x for x in ollama_models for q in FIRST_ROUND_QUESTIONS
]
type_list_col_answer, type_list_col_explanation

(['answer_granite4.1:30b', 'answer_ministral-3:14b', 'answer_llama3.1:8b'],
 ['explanation_granite4.1:30b',
  'explanation_ministral-3:14b',
  'explanation_llama3.1:8b'])

In [6]:
df_03_agents_analysis[type_list_col_answer].describe()

,answer_granite4.1:30b,answer_ministral-3:14b,answer_llama3.1:8b
count,726,726,726
unique,2,2,2
top,False,False,False
freq,665,661,711


In [7]:
df_03_agents_analysis[
    ["title", "is_reference"] + type_list_col_answer + type_list_col_explanation
].query("is_reference")

,title,is_reference,answer_granite4.1:30b,answer_ministral-3:14b,answer_llama3.1:8b,explanation_granite4.1:30b,explanation_ministral-3:14b,explanation_llama3.1:8b
15,A dynamic risk assessment method to address sa...,True,True,True,False,The paper meets the inclusion criteria by pres...,The paper meets all inclusion criteria by prop...,The paper does not explicitly mention a quanti...
23,A Marine GIS-based Alert System to Prevent Ves...,True,True,True,False,The paper presents a quantitative methodology ...,The study meets all inclusion criteria by prop...,The paper does not focus on quantitative risk ...
36,A Novel Method for Navigational Risk Assessmen...,True,True,True,False,The paper meets all inclusion criteria by pres...,The paper meets all inclusion criteria by prop...,The paper does not explicitly mention a quanti...
57,A System-Theory and Complex Network-Fused Appr...,True,False,False,False,The paper focuses on qualitative and quantitat...,While the paper addresses **allision** (collis...,The paper does not meet the inclusion criteria...
83,Allision risk analysis of offshore petroleum i...,True,True,True,True,The paper meets the inclusion criteria by perf...,The paper meets all inclusion criteria by focu...,The paper presents a quantitative risk assessm...
114,Assessing the impacts to vessel traffic from o...,True,False,False,False,The paper focuses on the impact of offshore wi...,While the paper addresses the **impact of offs...,The paper does not present a quantitative meth...
116,Assessing the validity of navigation risk asse...,True,False,True,False,The paper focuses on evaluating the accuracy o...,The paper meets **Inclusion Criteria 1** (quan...,The paper does not meet the inclusion criteria...
286,Evaluation of Ship Importance in Offshore Wind...,True,True,True,False,The study meets all inclusion criteria by pres...,The study meets all inclusion criteria by focu...,The paper analyzes the consequence of the coll...
341,Improving the coexistence of offshore wind far...,True,False,True,False,The paper focuses on a literature review and s...,The paper addresses **Goal 2 (safe distance be...,The paper does not specifically focus on the q...
387,Likelihood of a marine vessel accident from wi...,True,True,True,False,The paper meets all inclusion criteria: it qua...,The study meets all inclusion criteria by focu...,This paper does not meet the inclusion criteri...


In [8]:
# Reformat as bool
df_03_agents_analysis[type_list_col_answer] = (
    df_03_agents_analysis[type_list_col_answer]
    .where(
        df_03_agents_analysis[type_list_col_answer].isin(
            ["True", "False", "**True**", "**False**", True, False]
        ),
        pd.NA,
    )
    .replace({"True": True, "False": False, "**True**": True, "**False**": False})
)
df_03_agents_analysis[type_list_col_answer] = df_03_agents_analysis[
    type_list_col_answer
].astype(bool)
df_03_agents_analysis.dtypes

title_slug                         str
is_chosen_first_nra               bool
title                              str
doi                                str
type                               str
year                             int64
date                               str
abstract                           str
id_scopus                          str
keywords_scopus                    str
relevance_order_scopus         float64
is_reference                      bool
id_openalex                        str
relevance_openalex             float64
language_openalex                  str
pdf                                str
topic_openalex                     str
fields_openalex                    str
subfields_openalex                 str
domain_openalex                    str
keywords_openalex                  str
id_dimensions                      str
topic_dimensions                   str
keywords_dimensions                str
in_scopus                         bool
in_openalex              

In [9]:
df_seems_relevant = df_03_agents_analysis[
    df_03_agents_analysis[type_list_col_answer].any(axis=1, skipna=True)
]
len(df_seems_relevant)

75

In [10]:
df_seems_relevant[type_list_col_answer].corr()

,answer_granite4.1:30b,answer_ministral-3:14b,answer_llama3.1:8b
answer_granite4.1:30b,1.000000,-0.187907,0.153987
answer_ministral-3:14b,-0.187907,1.000000,0.196116
answer_llama3.1:8b,0.153987,0.196116,1.000000


In [42]:
plot_histogram(
    df_seems_relevant, col_name="topic_openalex", min_count_to_keep=1, height=900
)

In [11]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency


def phi_coefficient(series1, series2):
    contingency_table = pd.crosstab(series1, series2)
    chi2, _, _, _ = chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    phi = np.sqrt(chi2 / n)
    return phi


def ratio_coefficient(series1, series2):
    return (series1 == series2).sum() / len(series2)


def coefficiant_matrix(
    df_input: pd.DataFrame, col_names: list[str], coefficiant_func=phi_coefficient
):
    ret_matrix = pd.DataFrame(index=col_names, columns=col_names)

    for col1 in col_names:
        for col2 in col_names:
            ret_matrix.loc[col1, col2] = coefficiant_func(
                df_input[col1], df_input[col2]
            )
    return ret_matrix


# Example DataFrame
df = pd.DataFrame(
    {"A": [0, 1, 0, 1, 0, 1], "B": [0, 0, 1, 1, 0, 1], "C": [1, 1, 0, 0, 1, 0]}
)

# Compute phi correlation matrix

coefficiant_matrix(df, ["A", "B", "C"], coefficiant_func=ratio_coefficient)

,A,B,C
A,1.0,0.666667,0.333333
B,0.666667,1.0,0.0
C,0.333333,0.0,1.0


In [12]:
coef_matrix = coefficiant_matrix(df_03_agents_analysis, col_names=type_list_col_answer)
coef_matrix

,answer_granite4.1:30b,answer_ministral-3:14b,answer_llama3.1:8b
answer_granite4.1:30b,0.991051,0.783228,0.427219
answer_ministral-3:14b,0.783228,0.991551,0.446228
answer_llama3.1:8b,0.427219,0.446228,0.965963


In [13]:
import plotly.graph_objects as go


def plot_correlation_matrix(corr_matrix, title="Correlation Matrix", zmid=0):
    """
    Plot a correlation matrix as a heatmap using Plotly.

    Parameters:
    - corr_matrix: pandas DataFrame or numpy array, the correlation matrix to plot
    - title: str, title of the plot (default: "Correlation Matrix")
    """
    # Create the figure
    fig = go.Figure(
        data=go.Heatmap(
            z=corr_matrix.values if hasattr(corr_matrix, "values") else corr_matrix,
            x=corr_matrix.columns
            if hasattr(corr_matrix, "columns")
            else np.arange(corr_matrix.shape[1]),
            y=corr_matrix.index
            if hasattr(corr_matrix, "index")
            else np.arange(corr_matrix.shape[0]),
            colorscale="RdBu",
            zmid=zmid,
            text=corr_matrix.values if hasattr(corr_matrix, "values") else corr_matrix,
            texttemplate="%{text:.2f}",
            textfont={"size": 10},
            colorbar=dict(title="Correlation"),
        )
    )

    # Update layout for better visualization
    fig.update_layout(
        title=title,
        xaxis=dict(side="bottom"),
        width=1200,
        height=800,
        autotypenumbers="convert types",
    )

    # Make the plot square and adjust margins
    fig.update_yaxes(scaleanchor="x")
    fig.update_layout(margin=dict(l=100, r=100, b=100, t=100))
    return fig


# Example usage:
# import pandas as pd

plot_correlation_matrix(coef_matrix)

In [47]:
df_03_agents_analysis[df_03_agents_analysis["is_reference"]][type_list_col_answer]

,answer_granite4.1:30b,answer_ministral-3:14b,answer_llama3.1:8b
15,True,True,False
23,False,True,False
36,True,True,False
57,True,False,False
83,True,True,True
114,False,False,False
116,True,False,False
286,False,True,False
341,True,False,False
387,True,True,False


In [51]:
plot_correlation_matrix(
    coefficiant_matrix(
        df_03_agents_analysis[df_03_agents_analysis["is_reference"]],
        col_names=type_list_col_answer,
        coefficiant_func=ratio_coefficient,
    ),
    zmid=0.5,
)